In [ ]:
import os, random
from os.path import join as pjoin
from glob import glob
import numpy as np
import nibabel as nb
import scipy.io, scipy, scipy.stats
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.regression as sm
import seaborn as sns

base_dir = '/mnt/tambinidata/sleepstudy'
analysis_dir = pjoin(base_dir, 'analysis')
der_dir = pjoin(base_dir, 'data', 'derivatives', 'mriprep', 'fmriprep')
group_dir = pjoin(base_dir, 'data', 'derivatives', 'group')

In [ ]:
con_dir = pjoin(group_dir, 'ospan_vs_math')
ss_dirs = glob(pjoin(der_dir, 'sub-*'))
# ss_dirs
ss_list = []
for ss in ss_dirs:
    if 'html' in ss:
        a = 1
    else:
        ses_list = glob(pjoin(ss, 'ses-*'))
        pm_file1 = glob(pjoin(ss, 'ses-1', 'spm_os_model1_test', '*con_ospan_vs_math*'))
        # assert(len(pm_file1)==1)
        pm_file2 = glob(pjoin(ss, 'ses-3', 'spm_os_model1_test', '*con_ospan_vs_math*'))
        # assert(len(pm_file2)==1)

        am_file1 = glob(pjoin(ss, 'ses-2', 'spm_os_model1_test', '*con_ospan_vs_math*'))
        # assert(len(am_file1)==1)
        am_file2 = glob(pjoin(ss, 'ses-4', 'spm_os_model1_test', '*con_ospan_vs_math*'))
        # assert(len(am_file2)==1)

        if len(ses_list)==4 and len(am_file1)==1:
            if 'sub-107' not in ss and 'sub-127' not in ss and 'sub-136' not in ss and 'sub-145' not in ss and 'sub-153' not in ss and 'sub-160' not in ss:
                ss_list.append(ss)
print(len(ss_list))

In [ ]:
gm_mask = glob(pjoin(analysis_dir, 'gm_mask*'))
gm = nb.load(gm_mask[0]).get_data()>0.1

brain_mask = glob(pjoin(group_dir, 'MNI*ds_noCBS.*'))
mask = nb.load(brain_mask[0]).get_data()>0
gmmask = np.logical_and(mask, gm)

wm_file = glob(pjoin(group_dir, 'ospan_vs_math', 'allsessions', 'allsub', \
    'positive_noCBS','ospan_vs_math_allses_final.nii.gz'))
wm_file = glob(pjoin(analysis_dir, 'network_mask', 'network_7_ds.nii'))
print(wm_file)
wm_data = nb.load(wm_file[0]).get_data()>0
# gmmask = np.logical_and(gmmask, wm_data)

z = np.full((len(ss_list),2), np.nan)
z_btwn = np.full((len(ss_list),4), np.nan)
z_over_os = np.full(len(ss_list), np.nan)
Nsim = 100
z_null_os = np.full((len(ss_list), Nsim), np.nan)

con_name = '*con_ospan_vs_baseline*'

for iss, ss_dir in enumerate(ss_list):
    pm_file1 = glob(pjoin(ss_dir, 'ses-1', 'spm_os_model1_test', con_name))
    assert(len(pm_file1)==1)
    pm_patt1 = nb.load(pm_file1[0]).get_data()[gmmask]
    pm_file2 = glob(pjoin(ss_dir, 'ses-3', 'spm_os_model1_test', con_name))
    assert(len(pm_file2)==1)
    pm_patt2 = nb.load(pm_file2[0]).get_data()[gmmask]

    am_file1 = glob(pjoin(ss_dir, 'ses-2', 'spm_os_model1_test', con_name))
    assert(len(am_file1)==1)
    am_patt1 = nb.load(am_file1[0]).get_data()[gmmask]
    am_file2 = glob(pjoin(ss_dir, 'ses-4', 'spm_os_model1_test', con_name))
    assert(len(am_file2)==1)
    am_patt2 = nb.load(am_file2[0]).get_data()[gmmask]

    nan_idx_pm = np.logical_and(np.isnan(pm_patt1)==False, np.isnan(pm_patt2)==False)
    nan_idx_am = np.logical_and(np.isnan(am_patt1)==False, np.isnan(am_patt2)==False)
    nan_idx = np.logical_and(nan_idx_am, nan_idx_pm)

    if np.var(pm_patt1[nan_idx])>0 and np.var(pm_patt2[nan_idx])>0 \
        and np.var(am_patt1[nan_idx])>0 and np.var(am_patt2[nan_idx])>0:
        ov1 = pm_patt1[nan_idx]-am_patt1[nan_idx]
        ov2 = pm_patt2[nan_idx]-am_patt2[nan_idx]
        z_over_os[iss] = np.arctanh(scipy.stats.pearsonr(ov1,ov2)[0])
        z[iss,0] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], pm_patt2[nan_idx])[0])
        z[iss,1] = np.arctanh(scipy.stats.pearsonr(am_patt1[nan_idx], am_patt2[nan_idx])[0])

        z_btwn[iss, 0] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], am_patt1[nan_idx])[0])
        z_btwn[iss, 1] = np.arctanh(scipy.stats.pearsonr(pm_patt2[nan_idx], am_patt2[nan_idx])[0])
        z_btwn[iss, 2] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], am_patt2[nan_idx])[0])
        z_btwn[iss, 3] = np.arctanh(scipy.stats.pearsonr(pm_patt2[nan_idx], am_patt1[nan_idx])[0])

        for isim in range(Nsim):
            case = random.randint(0, 1)
            if case==0:
                ov1 = pm_patt1[nan_idx]-am_patt1[nan_idx]
            elif case==1:
                ov1 = am_patt1[nan_idx]-pm_patt1[nan_idx]

            case = random.randint(0, 1)
            if case==0:
                ov2 = pm_patt2[nan_idx]-am_patt2[nan_idx]
            elif case==1:
                ov2 = am_patt2[nan_idx]-pm_patt2[nan_idx]
            z_null_os[iss, isim] = np.arctanh(scipy.stats.pearsonr(ov1,ov2)[0])
            

In [ ]:

# z = np.full((len(ss_list),2), np.nan)
# z_btwn = np.full((len(ss_list),4), np.nan)
z_over_m = np.full(len(ss_list), np.nan)
z_null_m = np.full((len(ss_list), Nsim), np.nan)

con_name = '*con_math_vs_baseline*'

for iss, ss_dir in enumerate(ss_list):
    pm_file1 = glob(pjoin(ss_dir, 'ses-1', 'spm_os_model1_test', con_name))
    assert(len(pm_file1)==1)
    pm_patt1 = nb.load(pm_file1[0]).get_data()[gmmask]
    pm_file2 = glob(pjoin(ss_dir, 'ses-3', 'spm_os_model1_test', con_name))
    assert(len(pm_file2)==1)
    pm_patt2 = nb.load(pm_file2[0]).get_data()[gmmask]

    am_file1 = glob(pjoin(ss_dir, 'ses-2', 'spm_os_model1_test', con_name))
    assert(len(am_file1)==1)
    am_patt1 = nb.load(am_file1[0]).get_data()[gmmask]
    am_file2 = glob(pjoin(ss_dir, 'ses-4', 'spm_os_model1_test', con_name))
    assert(len(am_file2)==1)
    am_patt2 = nb.load(am_file2[0]).get_data()[gmmask]

    nan_idx_pm = np.logical_and(np.isnan(pm_patt1)==False, np.isnan(pm_patt2)==False)
    nan_idx_am = np.logical_and(np.isnan(am_patt1)==False, np.isnan(am_patt2)==False)
    nan_idx = np.logical_and(nan_idx_am, nan_idx_pm)

    if np.var(pm_patt1[nan_idx])>0 and np.var(pm_patt2[nan_idx])>0 \
        and np.var(am_patt1[nan_idx])>0 and np.var(am_patt2[nan_idx])>0:
        ov1 = pm_patt1[nan_idx]-am_patt1[nan_idx]
        ov2 = pm_patt2[nan_idx]-am_patt2[nan_idx]
        z_over_m[iss] = np.arctanh(scipy.stats.pearsonr(ov1,ov2)[0])
        # z[iss,0] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], pm_patt2[nan_idx])[0])
        # z[iss,1] = np.arctanh(scipy.stats.pearsonr(am_patt1[nan_idx], am_patt2[nan_idx])[0])

        # z_btwn[iss, 0] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], am_patt1[nan_idx])[0])
        # z_btwn[iss, 1] = np.arctanh(scipy.stats.pearsonr(pm_patt2[nan_idx], am_patt2[nan_idx])[0])
        # z_btwn[iss, 2] = np.arctanh(scipy.stats.pearsonr(pm_patt1[nan_idx], am_patt2[nan_idx])[0])
        # z_btwn[iss, 3] = np.arctanh(scipy.stats.pearsonr(pm_patt2[nan_idx], am_patt1[nan_idx])[0])

        for isim in range(Nsim):
            case = random.randint(0, 1)
            if case==0:
                ov1 = pm_patt1[nan_idx]-am_patt1[nan_idx]
            elif case==1:
                ov1 = am_patt1[nan_idx]-pm_patt1[nan_idx]

            case = random.randint(0, 1)
            if case==0:
                ov2 = pm_patt2[nan_idx]-am_patt2[nan_idx]
            elif case==1:
                ov2 = am_patt2[nan_idx]-pm_patt2[nan_idx]
            z_null_m[iss, isim] = np.arctanh(scipy.stats.pearsonr(ov1,ov2)[0])

In [ ]:
z_over = np.transpose(np.vstack((z_over_os, z_over_m)))
print(z_over.shape)
sns.barplot(data=z_over,ci=68)
print(scipy.stats.ttest_rel(z_over_m, z_over_os))
# scipy.stats.ttest_1samp(z_over, 0, nan_policy='omit')
# plt.figure()
# plt.hist(np.nanmean(z_null,axis=0))

In [ ]:
# sns.barplot(data=z_over,ci=68)
# scipy.stats.ttest_1samp(z_over, 0, nan_policy='omit')
plt.figure()
plt.hist(np.nanmean(z_null_m,axis=0))
plt.figure()
plt.hist(np.nanmean(z_null_os,axis=0))
print(np.mean(np.mean(z_null_os,axis=0)>=np.mean(z_over_os)))
print(np.mean(np.mean(z_null_m,axis=0)>=np.mean(z_over_m)))
print(np.mean(np.mean(z_null_os,axis=0)-np.mean(z_null_m,axis=0)>=np.mean(z_over_os-z_over_m)))


In [ ]:
ss_list

In [ ]:
np.nanmean(z,axis=0)
# z

In [ ]:
print(np.nanmean(z_btwn,axis=0))
print(np.nanmean(z_btwn))

In [ ]:
scipy.stats.ttest_rel(z[:,0], np.mean(z_btwn, axis=1), nan_policy='omit')

In [ ]:
scipy.stats.ttest_rel(z[:,1], np.mean(z_btwn[:,2:4], axis=1), nan_policy='omit')

In [ ]:
sns.barplot(data=z, ci=68)


In [ ]:
z[:,0]

In [ ]:
sns.barplot(data=np.mean(z_btwn,axis=1),ci=68)
